# VR Ramp Cells — Medial-Lateral Gradient in MEC

## What this notebook tests

**Ramp cells** fire at a progressively increasing or decreasing rate across a defined segment of
the VR track. They are classified using an OLS regression of the smoothed firing rate profile
against position within the outbound (30–90 cm) and homebound (110–170 cm) regions, compared
against a shuffle null.

- **Outbound ramp (+)**: firing rate increases as the animal runs toward the reward zone
- **Outbound ramp (−)**: firing rate decreases approaching the reward zone
- **Homebound ramp (+)**: firing rate increases as the animal runs back toward the start
- **Homebound ramp (−)**: firing rate decreases running back to the start

Classification is pre-computed and stored in `ramps.parquet` under each session's
`VR/tuning_scores/` directory.

## Question

Is the prevalence or sign of ramp coding distributed non-uniformly along the **medial-lateral
axis** of MEC? More lateral MEC is associated with larger spatial scale and potentially
different contextual coding strategies.

## Filters

| Filter | Value |
|--------|-------|
| Brain region | `brain_region` starts with `ENTm` |
| Cell type | Principal cells: `firing_rate_VR < 10 Hz` |
| Trial type | `TRIAL_TYPE` (default `'b'` — beaconed only) |
| ML split | `|SC_x| ≤ 3400 µm` → **Medial**; `> 3400 µm` → **Lateral** |

Analyses are shown for all MEC principal cells and separately for GC and NGS subpopulations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr, mannwhitneyu, chi2_contingency
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
source_path         = '/Users/harryclark/Downloads/COHORT12/'
CLASSIFICATIONS_CSV = '/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv'

TRIAL_TYPE = 'b'    # 'b' = beaconed, 'nb' = non-beaconed, 'b+nb' = combined
ML_SPLIT   = 3400   # medial / lateral boundary (µm)
ML_BINS    = 10     # ML position bins for gradient plots

mouse_days = {
    20: [14,15,16,17,18,19,20,21,22,23,24,25,26],
    21: [15,16,17,18,19,20,21,22,23,24,25,26],
    22: [33,34,35,36,37,38,39,40,41],
    25: [16,17,18,19,20,21,22,23,24,25],
    26: [11,12,13,14,15,16,17,18,19],
    27: [16,17,18,19,20,21,22,23,24,26],
    28: [16,17,18,19,20,21,22,23,25],
    29: [16,17,18,19,20,21,22,23,25],
}

# Colours
RAMP_COLOURS = {
    'outbound_only': '#26A69A',
    'homebound_only': '#42A5F5',
    'both':          '#7E57C2',
    'neither':       '#CFD8DC',
}
SIGN_COLOURS = {'+': '#E53935', '-': '#1565C0', '/': '#CFD8DC'}

In [ ]:
# ── Load pre-computed ramp.parquet files ──────────────────────────────────────
keep_cols = ['cluster_id','brain_region','trials','context',
             'outbound_slope','outbound_sig','outbound_sign',
             'homebound_slope','homebound_sig','homebound_sign']

all_ramps = []
for mouse, days in mouse_days.items():
    for day in days:
        p = (Path(source_path) / f'M{mouse}' / f'D{day:02}' /
             'VR' / 'tuning_scores' / 'ramps.parquet')
        if not p.exists():
            continue
        df_r = pd.read_parquet(p, columns=keep_cols)
        df_r['mouse'] = int(mouse)
        df_r['day']   = int(day)
        all_ramps.append(df_r)

df_ramps_all = pd.concat(all_ramps, ignore_index=True)
print(f'Loaded {len(df_ramps_all)} rows from {len(all_ramps)} sessions')

# Filter to chosen trial type
df_ramps = df_ramps_all[df_ramps_all['trials'] == TRIAL_TYPE].copy()
print(f'After trial_type == {TRIAL_TYPE!r}: {len(df_ramps)} rows')

# ── Merge with cell_classifications ───────────────────────────────────────────
df_class = pd.read_csv(CLASSIFICATIONS_CSV)[
    ['mouse','day','cluster_id','SC_x','cell_type','firing_rate_VR']
].copy()
df_class['ml']    = df_class['SC_x'].abs()
df_class['mouse'] = df_class['mouse'].astype(int)
df_class['day']   = df_class['day'].astype(int)

df = df_ramps.merge(df_class, on=['mouse','day','cluster_id'], how='inner')

# ── MEC + principal cells ─────────────────────────────────────────────────────
df = df[
    df['brain_region'].str.startswith('ENTm', na=False) &
    (df['firing_rate_VR'] < 10)
].copy()

# Derived columns
df['medial']            = df['ml'] <= ML_SPLIT
df['is_outbound_ramp']  = df['outbound_sig'].astype(bool)
df['is_homebound_ramp'] = df['homebound_sig'].astype(bool)
df['is_ramp']           = df['is_outbound_ramp'] | df['is_homebound_ramp']
df['ramp_type'] = np.where(
    df['is_outbound_ramp'] & df['is_homebound_ramp'], 'both',
    np.where(df['is_outbound_ramp'],  'outbound_only',
    np.where(df['is_homebound_ramp'], 'homebound_only', 'neither'))
)

print(f'\nMEC principal cells (ENTm*, FR<10Hz): {len(df)}')
print(df['cell_type'].value_counts().to_string())
print(f'\nRamp type breakdown:')
print(df['ramp_type'].value_counts().to_string())
print(f'\nAny ramp: {df["is_ramp"].sum()} / {len(df)}  ({100*df["is_ramp"].mean():.1f}%)')
print(f'Outbound:  {df["is_outbound_ramp"].sum()} / {len(df)}  ({100*df["is_outbound_ramp"].mean():.1f}%)')
print(f'Homebound: {df["is_homebound_ramp"].sum()} / {len(df)}  ({100*df["is_homebound_ramp"].mean():.1f}%)')

---
## Population overview

Overall ramp type proportions and sign breakdown before any ML stratification.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.0),
                         gridspec_kw=dict(wspace=0.42))

# A: ramp type pie / bar
ax = axes[0]
ramp_order = ['outbound_only','homebound_only','both','neither']
counts = df['ramp_type'].value_counts().reindex(ramp_order, fill_value=0)
bars = ax.bar(range(len(ramp_order)), counts.values,
              color=[RAMP_COLOURS[r] for r in ramp_order], alpha=0.85, lw=0)
ax.set_xticks(range(len(ramp_order)))
ax.set_xticklabels([r.replace('_','-\n') for r in ramp_order], fontsize=8)
ax.set_ylabel('N cells', fontsize=9)
ax.set_title(f'Ramp type counts  (trial_type={TRIAL_TYPE!r})',
             fontsize=9, fontweight='bold')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            str(val), ha='center', va='bottom', fontsize=8)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

# B: outbound sign breakdown
ax = axes[1]
ob_sig = df[df['is_outbound_ramp']]
ob_counts = ob_sig['outbound_sign'].value_counts().reindex(['+','-'], fill_value=0)
ax.bar([0, 1], ob_counts.values,
       color=[SIGN_COLOURS['+'], SIGN_COLOURS['-']], alpha=0.85, lw=0)
ax.set_xticks([0, 1])
ax.set_xticklabels(['+ (increasing)', '− (decreasing)'], fontsize=8)
ax.set_ylabel('N cells', fontsize=9)
ax.set_title(f'Outbound ramp sign\n(n={len(ob_sig)} sig cells)', fontsize=9, fontweight='bold')
for xi, val in enumerate(ob_counts.values):
    ax.text(xi, val+0.5, str(val), ha='center', va='bottom', fontsize=9)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

# C: homebound sign breakdown
ax = axes[2]
hb_sig = df[df['is_homebound_ramp']]
hb_counts = hb_sig['homebound_sign'].value_counts().reindex(['+','-'], fill_value=0)
ax.bar([0, 1], hb_counts.values,
       color=[SIGN_COLOURS['+'], SIGN_COLOURS['-']], alpha=0.85, lw=0)
ax.set_xticks([0, 1])
ax.set_xticklabels(['+ (increasing)', '− (decreasing)'], fontsize=8)
ax.set_ylabel('N cells', fontsize=9)
ax.set_title(f'Homebound ramp sign\n(n={len(hb_sig)} sig cells)', fontsize=9, fontweight='bold')
for xi, val in enumerate(hb_counts.values):
    ax.text(xi, val+0.5, str(val), ha='center', va='bottom', fontsize=9)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle('MEC principal cells — ramp cell overview', fontsize=11, fontweight='bold')
plt.savefig('ramp_overview.pdf', bbox_inches='tight', dpi=150)
plt.show()

---
## Medial-lateral gradient analysis

For each cell-type group (all / GC / NGS):

**Figure 1 (2×2)**
- A: Fraction any-ramp cells per ML bin, with ramp-type composition (stacked)
- B: Outbound ramp fraction per ML bin, coloured by sign (+/−)
- C: Homebound ramp fraction per ML bin, coloured by sign (+/−)
- D: Per-mouse fraction ramp — medial vs lateral

**Figure 2 (1×2)**
- A: Outbound slope distribution — medial vs lateral (significant cells only)
- B: Homebound slope distribution — medial vs lateral (significant cells only)

In [ ]:
CELL_TYPE_GROUPS = [
    ('all',  'All MEC principal',  df),
    ('GC',   'Grid cells (GC)',    df[df['cell_type'] == 'GC']),
    ('NGS',  'Non-grid spatial',   df[df['cell_type'] == 'NGS']),
]

for ct_key, ct_label, dg in CELL_TYPE_GROUPS:
    if len(dg) == 0:
        print(f'No cells for {ct_label} — skipping')
        continue

    med = dg[dg['medial']]
    lat = dg[~dg['medial']]

    # Stats
    rho_any,  p_any  = spearmanr(dg['ml'], dg['is_ramp'].astype(float))
    rho_ob,   p_ob   = spearmanr(dg['ml'], dg['is_outbound_ramp'].astype(float))
    rho_hb,   p_hb   = spearmanr(dg['ml'], dg['is_homebound_ramp'].astype(float))
    _, mwu_any        = mannwhitneyu(med['is_ramp'].astype(float),
                                     lat['is_ramp'].astype(float),
                                     alternative='two-sided') if len(med)>1 and len(lat)>1                         else (np.nan, np.nan)
    # Chi-squared ramp vs no-ramp, medial vs lateral
    ct_tab = pd.crosstab(dg['medial'], dg['is_ramp'])
    chi2, chi_p, _, _ = chi2_contingency(ct_tab) if ct_tab.shape == (2,2) else (np.nan,np.nan,None,None)

    print(f'\n── {ct_label}  (n={len(dg)}, ramp={dg["is_ramp"].sum()}) ──')
    print(f'  Spearman r(ML, is_ramp):     rho={rho_any:.3f}  p={p_any:.4f}')
    print(f'  Spearman r(ML, outbound):    rho={rho_ob:.3f}  p={p_ob:.4f}')
    print(f'  Spearman r(ML, homebound):   rho={rho_hb:.3f}  p={p_hb:.4f}')
    print(f'  Chi2 ramp vs ML split:       chi2={chi2:.2f}  p={chi_p:.4f}')
    print(f'  Medial  — any ramp: {med["is_ramp"].mean()*100:.1f}%  '
          f'outbound: {med["is_outbound_ramp"].mean()*100:.1f}%  '
          f'homebound: {med["is_homebound_ramp"].mean()*100:.1f}%')
    print(f'  Lateral — any ramp: {lat["is_ramp"].mean()*100:.1f}%  '
          f'outbound: {lat["is_outbound_ramp"].mean()*100:.1f}%  '
          f'homebound: {lat["is_homebound_ramp"].mean()*100:.1f}%')

    ml_vals = dg['ml'].values
    edges   = np.linspace(ml_vals.min(), ml_vals.max(), ML_BINS + 1)
    ctrs    = (edges[:-1] + edges[1:]) / 2.0
    n_all   = np.array([np.sum((ml_vals>=e0)&(ml_vals<e1)) for e0,e1 in zip(edges[:-1],edges[1:])])

    def frac_in_bin(mask_series):
        mv = mask_series.values.astype(float)
        return np.array([
            mv[(ml_vals>=e0)&(ml_vals<e1)].mean() if np.sum((ml_vals>=e0)&(ml_vals<e1))>0 else np.nan
            for e0,e1 in zip(edges[:-1],edges[1:])
        ])

    def count_in_bin(mask_series):
        mv = mask_series.values.astype(float)
        return np.array([mv[(ml_vals>=e0)&(ml_vals<e1)].sum()
                         for e0,e1 in zip(edges[:-1],edges[1:])])

    frac_any = frac_in_bin(dg['is_ramp'])
    frac_ob  = frac_in_bin(dg['is_outbound_ramp'])
    frac_hb  = frac_in_bin(dg['is_homebound_ramp'])

    # ── Figure 1: 2×2 ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(11, 8),
                             gridspec_kw=dict(hspace=0.45, wspace=0.40))

    # A: stacked ramp type per ML bin
    ax = axes[0, 0]
    width = np.diff(edges) * 0.8
    n_ob_only  = count_in_bin(dg['ramp_type'].eq('outbound_only'))
    n_hb_only  = count_in_bin(dg['ramp_type'].eq('homebound_only'))
    n_both     = count_in_bin(dg['ramp_type'].eq('both'))
    tot_ramp   = n_ob_only + n_hb_only + n_both
    p_ob_only  = np.where(n_all>0, n_ob_only/n_all, np.nan)
    p_hb_only  = np.where(n_all>0, n_hb_only/n_all, np.nan)
    p_both     = np.where(n_all>0, n_both/n_all, np.nan)
    ax.bar(ctrs, p_ob_only, width, color=RAMP_COLOURS['outbound_only'],
           alpha=0.85, lw=0, label='outbound only')
    ax.bar(ctrs, p_hb_only, width, bottom=p_ob_only,
           color=RAMP_COLOURS['homebound_only'], alpha=0.85, lw=0, label='homebound only')
    ax.bar(ctrs, p_both, width, bottom=p_ob_only+p_hb_only,
           color=RAMP_COLOURS['both'], alpha=0.85, lw=0, label='both')
    ax.axvline(ML_SPLIT, color='#455A64', lw=1.0, ls='--', alpha=0.6)
    axb = ax.twinx()
    axb.step(edges, np.append(n_all,n_all[-1]), color='#90A4AE', lw=0.9, where='post', alpha=0.6)
    axb.set_ylabel('N cells', fontsize=7, color='#90A4AE')
    axb.tick_params(labelsize=6, labelcolor='#90A4AE')
    axb.spines[['top','left']].set_visible(False)
    ax.set_xlabel('|SC_x|  (medial → lateral, µm)', fontsize=9)
    ax.set_ylabel('Fraction ramp cells', fontsize=9)
    ax.set_title(f'Ramp type composition across ML axis\n'
                 f'Spearman rho(ML, any ramp)={rho_any:.2f}  p={p_any:.4f}',
                 fontsize=8.5, fontweight='bold')
    ax.legend(fontsize=7, frameon=False, loc='upper right')
    ax.set_ylim(0, 1.05)
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

    # B: outbound ramp fraction + sign per ML bin
    ax = axes[0, 1]
    # For sign breakdown: among sig cells in each bin, count +/-
    n_ob_pos = count_in_bin((dg['outbound_sign'] == '+') & dg['is_outbound_ramp'])
    n_ob_neg = count_in_bin((dg['outbound_sign'] == '-') & dg['is_outbound_ramp'])
    p_ob_pos = np.where(n_all>0, n_ob_pos/n_all, np.nan)
    p_ob_neg = np.where(n_all>0, n_ob_neg/n_all, np.nan)
    ax.bar(ctrs, p_ob_pos, width, color=SIGN_COLOURS['+'], alpha=0.80, lw=0, label='outbound +')
    ax.bar(ctrs, p_ob_neg, width, bottom=p_ob_pos,
           color=SIGN_COLOURS['-'], alpha=0.80, lw=0, label='outbound −')
    ax.axvline(ML_SPLIT, color='#455A64', lw=1.0, ls='--', alpha=0.6)
    sem_ob = np.where(n_all>0, np.sqrt(frac_ob*(1-frac_ob)/np.where(n_all>0,n_all,1)), np.nan)
    ax.errorbar(ctrs, frac_ob, yerr=sem_ob, fmt='none', color='#37474F', lw=1.0, capsize=3)
    ax.set_xlabel('|SC_x|  (medial → lateral, µm)', fontsize=9)
    ax.set_ylabel('Fraction outbound ramp', fontsize=9)
    ax.set_title(f'Outbound ramp (sign breakdown)\n'
                 f'rho={rho_ob:.2f}  p={p_ob:.4f}',
                 fontsize=8.5, fontweight='bold')
    ax.legend(fontsize=7.5, frameon=False)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.1 + 0.02)
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

    # C: homebound ramp fraction + sign per ML bin
    ax = axes[1, 0]
    n_hb_pos = count_in_bin((dg['homebound_sign'] == '+') & dg['is_homebound_ramp'])
    n_hb_neg = count_in_bin((dg['homebound_sign'] == '-') & dg['is_homebound_ramp'])
    p_hb_pos = np.where(n_all>0, n_hb_pos/n_all, np.nan)
    p_hb_neg = np.where(n_all>0, n_hb_neg/n_all, np.nan)
    ax.bar(ctrs, p_hb_pos, width, color=SIGN_COLOURS['+'], alpha=0.80, lw=0, label='homebound +')
    ax.bar(ctrs, p_hb_neg, width, bottom=p_hb_pos,
           color=SIGN_COLOURS['-'], alpha=0.80, lw=0, label='homebound −')
    ax.axvline(ML_SPLIT, color='#455A64', lw=1.0, ls='--', alpha=0.6)
    sem_hb = np.where(n_all>0, np.sqrt(frac_hb*(1-frac_hb)/np.where(n_all>0,n_all,1)), np.nan)
    ax.errorbar(ctrs, frac_hb, yerr=sem_hb, fmt='none', color='#37474F', lw=1.0, capsize=3)
    ax.set_xlabel('|SC_x|  (medial → lateral, µm)', fontsize=9)
    ax.set_ylabel('Fraction homebound ramp', fontsize=9)
    ax.set_title(f'Homebound ramp (sign breakdown)\n'
                 f'rho={rho_hb:.2f}  p={p_hb:.4f}',
                 fontsize=8.5, fontweight='bold')
    ax.legend(fontsize=7.5, frameon=False)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.1 + 0.02)
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

    # D: per-mouse fraction ramp medial vs lateral
    ax = axes[1, 1]
    mice  = sorted(dg['mouse'].unique())
    x_med, x_lat = [], []
    for mouse in mice:
        sub = dg[dg['mouse'] == mouse]
        x_med.append(sub[sub['medial']]['is_ramp'].mean())
        x_lat.append(sub[~sub['medial']]['is_ramp'].mean())
    x = np.arange(len(mice)); w = 0.35
    ax.bar(x-w/2, x_med, w, color='#5C6BC0', alpha=0.80, lw=0, label=f'Medial ≤{ML_SPLIT}')
    ax.bar(x+w/2, x_lat, w, color='#EF5350', alpha=0.80, lw=0, label=f'Lateral >{ML_SPLIT}')
    for xi, fm, fl in zip(x, x_med, x_lat):
        if np.isfinite(fm) and np.isfinite(fl):
            ax.plot([xi-w/2, xi+w/2], [fm, fl], color='k', lw=0.8, alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels([f'M{m}' for m in mice], fontsize=7)
    ax.set_ylabel('Fraction ramp cells', fontsize=9)
    ax.set_title(f'Per-mouse: medial vs lateral\nchi2 p={chi_p:.4f}',
                 fontsize=8.5, fontweight='bold')
    ax.legend(fontsize=7.5, frameon=False)
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

    fig.suptitle(f'MEC {ct_label} (FR<10Hz) — ramp cell ML gradient  (trial_type={TRIAL_TYPE!r})',
                 fontsize=10.5, fontweight='bold')
    plt.savefig(f'ramp_ML_gradient_{ct_key}.pdf', bbox_inches='tight', dpi=150)
    plt.show()

    # ── Figure 2: slope distributions ─────────────────────────────────────────
    ob_sig  = dg[dg['is_outbound_ramp']]
    hb_sig  = dg[dg['is_homebound_ramp']]

    if len(ob_sig) < 2 and len(hb_sig) < 2:
        print(f'  Too few ramp cells for slope distributions — skipping Figure 2')
        continue

    fig2, axes2 = plt.subplots(1, 2, figsize=(9, 3.8),
                               gridspec_kw=dict(wspace=0.40))

    for ax, seg_df, slope_col, seg_label in [
        (axes2[0], ob_sig, 'outbound_slope',  'Outbound'),
        (axes2[1], hb_sig, 'homebound_slope', 'Homebound'),
    ]:
        med_s = seg_df[seg_df['medial']][slope_col].dropna()
        lat_s = seg_df[~seg_df['medial']][slope_col].dropna()
        _, mwu_p_s = mannwhitneyu(med_s, lat_s, alternative='two-sided')                      if len(med_s)>1 and len(lat_s)>1 else (np.nan, np.nan)
        lim = np.nanpercentile(np.abs(seg_df[slope_col].dropna()), 98)
        bins_s = np.linspace(-lim, lim, 30)
        ax.hist(med_s, bins=bins_s, color='#5C6BC0', alpha=0.70, density=True,
                edgecolor='none', label=f'Medial (n={len(med_s)})')
        ax.hist(lat_s, bins=bins_s, color='#EF5350', alpha=0.70, density=True,
                edgecolor='none', label=f'Lateral (n={len(lat_s)})')
        ax.axvline(med_s.median(), color='#5C6BC0', lw=1.5, ls='--', alpha=0.9)
        ax.axvline(lat_s.median(), color='#EF5350', lw=1.5, ls='--', alpha=0.9)
        ax.axvline(0, color='#90A4AE', lw=0.8, ls=':')
        ax.set_xlabel(f'{seg_label} slope  (Hz/cm)', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.set_title(f'{seg_label} ramp slope\nMann-Whitney p={mwu_p_s:.4f}',
                     fontsize=9, fontweight='bold')
        ax.legend(fontsize=7.5, frameon=False)
        ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

    fig2.suptitle(f'MEC {ct_label} — ramp slope distributions: medial vs lateral',
                  fontsize=10.5, fontweight='bold')
    plt.savefig(f'ramp_slope_ML_{ct_key}.pdf', bbox_inches='tight', dpi=150)
    plt.show()